# 02 — Spraying Stage Explorer

Spray solution (acetone + ethylcellulose) is atomised onto the heated particle bed.
Acetone evaporates; coating polymer deposits on particle surfaces.

**Move the sliders** to explore how spray conditions affect:
- Product temperature (heat balance between inlet air and solvent evaporation)
- Acetone content on particles (accumulation vs. drying rate)
- Cumulative coating weight gain (%)


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from fluid_bed.parameters import ProcessParameters
from fluid_bed.models.spraying import run_spraying

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})

In [2]:
FIXED = dict(
    particle_density = 1000.0,
    cp_particle      = 1400.0,
    diameter_bed     = 0.60,
    rho_air          = 1.10,
    cp_air           = 1010.0,
)
print("Fixed properties loaded.")


Fixed properties loaded.


In [ ]:
S = dict(continuous_update=False)

_controls02 = {
    "T_inlet_C":    widgets.FloatSlider(70.0, min=40,    max=100,  step=5,   description="T inlet (°C)",    **S),
    "flow_m3h":     widgets.FloatSlider(300,  min=100,   max=600,  step=25,  description="Air flow (m³/h)", **S),
    "spray_g_min":  widgets.FloatSlider(15.0, min=5,     max=50,   step=1,   description="Spray (g/min)",   **S),
    "dmc_pct":      widgets.FloatSlider(15.0, min=5,     max=30,   step=1,   description="Dry matter (%)",  **S),
    "atom_m3h":     widgets.FloatSlider(20.0, min=0,     max=80,   step=5,   description="Atom. air (m³/h)",**S),
    "T_init_C":     widgets.FloatSlider(55.0, min=20,    max=80,   step=5,   description="T0 product (C)",  **S),
    "duration_min": widgets.FloatSlider(30.0, min=5,     max=90,   step=5,   description="Duration (min)",  **S),
    "batch_kg":     widgets.FloatSlider(2.0,  min=0.5,   max=5.0,  step=0.5, description="Batch (kg)",      **S),
    "r_spr_1e6":    widgets.FloatSlider(6.72, min=0,     max=20.0, step=0.1, description="r_spray (x1e-6)", **S),
}


def _run02(T_inlet_C, flow_m3h, spray_g_min, dmc_pct,
           atom_m3h, T_init_C, duration_min, batch_kg, r_spr_1e6):
    rho   = FIXED["rho_air"]
    m_air = (flow_m3h / 3600.0) * rho
    m_atm = (atom_m3h / 3600.0) * rho
    T_inlet_K = T_inlet_C + 273.15
    T_init_K  = T_init_C  + 273.15

    params = ProcessParameters(
        **FIXED,
        diameter_eq          = 200e-6,
        batch_size           = batch_kg,
        air_flow_rates       = (m_air, m_air, m_air),
        air_temperatures     = (T_inlet_K, T_inlet_K, T_inlet_K),
        spray_rate           = spray_g_min / 60.0 / 1000.0,
        dry_matter_conc      = dmc_pct / 100.0,
        atomization_air_flow = m_atm,
        r_spraying           = r_spr_1e6 * 1e-6,
    )

    res = run_spraying(params, duration=duration_min * 60.0, T_particle_init=T_init_K)
    t_min       = res.t / 60.0
    T_p_C       = res.T_particle - 273.15
    T_g_C       = res.T_gas      - 273.15
    acetone_pct = res.Y_particle * 100.0
    wg_pct      = res.M_coating / batch_kg * 100.0

    spray_dm_total = spray_g_min / 60.0 / 1000.0 * dmc_pct / 100.0 * duration_min * 60.0
    efficiency = res.M_coating[-1] / spray_dm_total * 100.0 if spray_dm_total > 0 else 0.0

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    ax = axes[0]
    ax.plot(t_min, T_g_C, "r--", lw=1.5, label="Gas T_g")
    ax.plot(t_min, T_p_C, "b-",  lw=2.5, label="Product T_p")
    ax.set_xlabel("Time (min)"); ax.set_ylabel("Temperature (C)")
    ax.set_title("Temperatures"); ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(t_min, acetone_pct, color="darkorange", lw=2.5)
    ax.set_xlabel("Time (min)"); ax.set_ylabel("Acetone on particles (wt %)")
    ax.set_title("Particle acetone content"); ax.grid(True, alpha=0.3)

    ax = axes[2]
    ax.plot(t_min, wg_pct, color="mediumpurple", lw=2.5, label="Model")
    if r_spr_1e6 > 0:
        t_ideal = np.linspace(0, duration_min, 200)
        wg_ideal_line = (spray_g_min / 60.0 / 1000.0 * dmc_pct / 100.0
                         * t_ideal * 60.0 / batch_kg * 100.0)
        ax.plot(t_ideal, wg_ideal_line, "k--", lw=1, alpha=0.4, label="100% efficiency")
        ax.legend()
    ax.set_xlabel("Time (min)"); ax.set_ylabel("Weight gain (%)")
    ax.set_title("Cumulative coating deposition"); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    plt.close()
    print(f"T_product = {T_p_C[-1]:.1f} C  |  "
          f"Acetone = {acetone_pct[-1]:.5f} %  |  "
          f"Weight gain = {wg_pct[-1]:.3f}%  |  "
          f"Coating efficiency = {efficiency:.1f}%")


_out02 = widgets.interactive_output(_run02, _controls02)
clear_output(wait=True)
display(widgets.VBox(list(_controls02.values())), _out02)